## Import Library

In [ ]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI
from google.colab import userdata

## Loading API Keys

In [ ]:
# Setting up
import os
from google.colab import userdata

load_dotenv(override=True)
# Retrieve your OpenRouter API key from Colab secrets
openrouter_api_key = userdata.get('OPENROUTER_API_KEY')

if openrouter_api_key:
    print(f"OpenRouter API Key retrieved from Colab secrets (begins {openrouter_api_key[:8]})")
else:
    print("Please add your OpenRouter API key to Colab secrets as 'OPENROUTER_API_KEY'.")
    # Exit or raise error if API key is not found to prevent further errors
    # For now, we'll proceed but be aware it might fail later

MODEL = "google/gemma-3-4b-it:free" # Setting the user's preferred model here
openai = OpenAI(api_key=openrouter_api_key, base_url="https://openrouter.ai/api/v1")

Once the OpenAI client is successfully initialized with your OpenRouter API key, you can make API calls. Here's an example of how to interact with a free model (e.g., `nousresearch/nous-hermes-2-mixtral-8x7b-dpo`) via OpenRouter:

In [ ]:
try:
    # Ensure the client is initialized from the previous cell
    if 'openai' not in locals():
        print("OpenAI client not initialized. Please run the previous cell first.")
    else:
        messages = [
            {"role": "user", "content": "What is the capital of France?"}
        ]

        # You can specify any model available on OpenRouter
        # 'google/gemma-7b-it' is a commonly available free model
        chat_completion = openai.chat.completions.create(
            model="google/gemma-3-4b-it:free",
            messages=messages
        )

        print(chat_completion.choices[0].message.content)
except Exception as e:
    print(f"An error occurred: {e}")

## Creating Sample Employee Data

I will create a `knowledge-base/employees/` directory and populate it with some sample employee information files. This will allow the RAG application to retrieve relevant information based on queries.

In [ ]:
import os

employees_dir = 'knowledge-base/employees'
os.makedirs(employees_dir, exist_ok=True)

# Sample employee data
emp1_content = "Name: Alice Smith\nTitle: Software Engineer\nDepartment: Engineering\nEmail: alice.smith@example.com\nSkills: Python, Java, AWS, Machine Learning"

emp2_content = "Name: Bob Johnson\nTitle: Project Manager\nDepartment: Product Development\nEmail: bob.johnson@example.com\nSkills: Agile, Scrum, Product Roadmapping, Team Leadership"

emp3_content = "Name: Carol White\nTitle: HR Specialist\nDepartment: Human Resources\nEmail: carol.white@example.com\nSkills: Recruitment, Employee Relations, Benefits Administration, Conflict Resolution"

with open(os.path.join(employees_dir, 'employee Alice Smith.txt'), 'w') as f:
    f.write(emp1_content)

with open(os.path.join(employees_dir, 'employee Bob Johnson.txt'), 'w') as f:
    f.write(emp2_content)

with open(os.path.join(employees_dir, 'employee Carol White.txt'), 'w') as f:
    f.write(emp3_content)

print(f"Created directory '{employees_dir}' and populated it with employee data.")

## Loading Data into `knowledge`

Now that the sample employee files are created, I will re-run the data loading cell to populate the `knowledge` dictionary.

In [ ]:
knowledge = {}

filenames = glob.glob("knowledge-base/employees/*")

for filename in filenames:
    name = Path(filename).stem.replace('employee ', '')
    with open(filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

# Display the loaded knowledge to verify
display(knowledge)

## Creating Sample Product Data

I will create a `knowledge-base/products/` directory and populate it with some sample product information files to demonstrate how a product knowledge base might look.

In [ ]:
import os

products_dir = 'knowledge-base/products'
os.makedirs(products_dir, exist_ok=True)

# Sample product data
prod1_content = "Product Name: Laptop Pro X\nCategory: Electronics\nPrice: $1200\nFeatures: 16GB RAM, 512GB SSD, Intel i7 Processor, 14-inch Retina Display\nDescription: High-performance laptop for professionals."

prod2_content = "Product Name: Ergonomic Office Chair\nCategory: Office Furniture\nPrice: $350\nFeatures: Adjustable lumbar support, breathable mesh, 360-degree swivel\nDescription: Comfortable and supportive chair for long working hours."

prod3_content = "Product Name: Wireless Earbuds Elite\nCategory: Audio\nPrice: $150\nFeatures: Noise-cancelling, 10-hour battery life, Waterproof, Bluetooth 5.2\nDescription: Premium sound quality with active noise cancellation."

with open(os.path.join(products_dir, 'product Laptop Pro X.txt'), 'w') as f:
    f.write(prod1_content)

with open(os.path.join(products_dir, 'product Ergonomic Office Chair.txt'), 'w') as f:
    f.write(prod2_content)

with open(os.path.join(products_dir, 'product Wireless Earbuds Elite.txt'), 'w') as f:
    f.write(prod3_content)

print(f"Created directory '{products_dir}' and populated it with product data.")

## Loading Product Data into `product_knowledge`

Now that the sample product files are created, I will load them into a new dictionary called `product_knowledge`.

In [ ]:
product_knowledge = {}

product_filenames = glob.glob("knowledge-base/products/*")

for filename in product_filenames:
    name = Path(filename).stem.replace('product ', '')
    with open(filename, "r", encoding="utf-8") as f:
        product_knowledge[name.lower()] = f.read()

# Display the loaded product knowledge to verify
display(product_knowledge)

In [ ]:
knowledge

## System Prefix

In [ ]:
SYSTEM_PREFIX = """
You represent Martllm, the Super Mart company.
You are an expert in answering questions about Martllm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

## Fetching Data

In [ ]:
def get_relevant_context(message):
    message_lower = message.lower()
    relevant_docs = []

    # Check employee knowledge
    for name, doc in knowledge.items():
        if name in message_lower:
            relevant_docs.append(doc)

    # Check product knowledge
    for product_name, doc in product_knowledge.items():
        if product_name in message_lower:
            relevant_docs.append(doc)

    return relevant_docs

In [ ]:
get_relevant_context("Who is Bob Johnson?")

In [ ]:
def chat(message, history):
    relevant_docs = get_relevant_context(message)
    context_string = "\n".join(relevant_docs)
    system_message = SYSTEM_PREFIX + context_string
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

## Gradio Part

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)

# Day 2

In [ ]:
!pip install -qU langchain-openai langchain-chroma langchain-huggingface langchain-community

import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_core.documents import Document

In [ ]:
from langchain_openai import OpenAIEmbeddings

# Ensure openrouter_api_key is loaded from the earlier cell
# If it's not, you might need to re-run the 'Loading API Keys' section.

# Initialize OpenAIEmbeddings to use OpenRouter's API
openrouter_embeddings = OpenAIEmbeddings(
    openai_api_base="https://openrouter.ai/api/v1",
    openai_api_key=openrouter_api_key
)

print("OpenAIEmbeddings configured to use OpenRouter.")
# You can test it with a sample text:
# embedding_vector = openrouter_embeddings.embed_query("This is a test sentence.")
# print(f"Embedding vector length: {len(embedding_vector)}")

In [ ]:
MODEL = "google/gemma-3-4b-it:free"
db_name = "vector_db"
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")